In [2]:
%pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 17.1 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import duckdb

customers = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3],
    "city": ["Johor Bahru", "Kuala Lumpur", "Penang", "Ipoh", "Melaka", "Kuching"],
    "updated_at": ["2026-01-05", "2026-03-10", "2026-02-01",
                   "2026-04-02", "2026-01-20", "2026-05-15"]
})

duckdb.sql("SELECT * FROM customers").show()

┌─────────────┬──────────────┬────────────┐
│ customer_id │     city     │ updated_at │
│    int64    │   varchar    │  varchar   │
├─────────────┼──────────────┼────────────┤
│           1 │ Johor Bahru  │ 2026-01-05 │
│           1 │ Kuala Lumpur │ 2026-03-10 │
│           1 │ Penang       │ 2026-02-01 │
│           2 │ Ipoh         │ 2026-04-02 │
│           2 │ Melaka       │ 2026-01-20 │
│           3 │ Kuching      │ 2026-05-15 │
└─────────────┴──────────────┴────────────┘



In [5]:
duckdb.sql("""
SELECT customer_id, city, updated_at
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY customer_id
           ORDER BY updated_at ASC
         ) AS rn
  FROM customers
)
WHERE rn = 1
""").show()

┌─────────────┬─────────────┬────────────┐
│ customer_id │    city     │ updated_at │
│    int64    │   varchar   │  varchar   │
├─────────────┼─────────────┼────────────┤
│           2 │ Melaka      │ 2026-01-20 │
│           1 │ Johor Bahru │ 2026-01-05 │
│           3 │ Kuching     │ 2026-05-15 │
└─────────────┴─────────────┴────────────┘



In [9]:
transactions = pd.DataFrame({
    "txn_id": [101, 102, 103, 104],
    "product_id": [1, 2, 1, 9],
    "litres": [30, 25, 40, 20]
})

products = pd.DataFrame({
    "product_id": [1, 2],
    "product_name": ["RON95", "Diesel"]
})

duckdb.sql("""
SELECT t.*
FROM transactions t
LEFT JOIN products p
  ON t.product_id = p.product_id
WHERE p.product_id IS NULL
""").show()

┌────────┬────────────┬────────┐
│ txn_id │ product_id │ litres │
│ int64  │   int64    │ int64  │
├────────┼────────────┼────────┤
│    104 │          9 │     20 │
└────────┴────────────┴────────┘



In [10]:
duckdb.sql("""
SELECT p.product_name,
       COUNT(*) AS num_txns,
       SUM(t.litres) AS total_litres
FROM transactions t
LEFT JOIN products p
  ON t.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_litres DESC
""").show()

┌──────────────┬──────────┬──────────────┐
│ product_name │ num_txns │ total_litres │
│   varchar    │  int64   │    int128    │
├──────────────┼──────────┼──────────────┤
│ RON95        │        2 │           70 │
│ Diesel       │        1 │           25 │
│ NULL         │        1 │           20 │
└──────────────┴──────────┴──────────────┘

